# Prerequisites

This notebook runs inside the `quay.io/jupyter/pyspark-notebook` container
described in `lab_pyspark_jupyter_docker.md`. The working directory is
`/home/jovyan/work`, bind-mounted to the host project directory, so anything
written here survives a container restart.

**Two deviations from the notebook as handed out:**

1. The original cells read `/content/around_the_world_in_80_days.txt`.
   `/content` is the Google Colab scratch directory and does not exist in this
   image, so paths here are relative to the notebook instead.
2. The original downloads only the English book but then builds `rdd_2` from a
   French file that is never fetched. The `wget` cell below fetches both.

In [1]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8
# the French edition: needed by rdd_2, and used for the comparison at the end
!wget -nc -O le_tour_du_monde_en_quatre_vingts_jours.txt https://www.gutenberg.org/ebooks/46541.txt.utf-8
!ls -lh *.txt

--2026-09-09 09:16:56--  https://www.gutenberg.org/ebooks/103.txt.utf-8


Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 152.19.134.47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... 

connected.


HTTP request sent, awaiting response... 

302 Found
Location: http://www.gutenberg.org/cache/epub/103/pg103.txt [following]
URL transformed to HTTPS due to an HSTS policy
--2026-09-09 09:16:57--  https://www.gutenberg.org/cache/epub/103/pg103.txt
Reusing existing connection to www.gutenberg.org:443.
HTTP request sent, awaiting response... 

...

       le_tour_du_m  42%[=======>            ] 197.90K   275KB/s               

      le_tour_du_mo  56%[==========>         ] 261.65K   279KB/s               

     le_tour_du_mon  94%[=================>  ] 436.96K   364KB/s               

le_tour_du_monde_en 100%[===================>] 461.65K   353KB/s    in 1.3s    

2026-09-09 09:17:01 (353 KB/s) - ‘le_tour_du_monde_en_quatre_vingts_jours.txt’ saved [472731/472731]



-rw-r--r-- 1 jovyan jovyan 395K Sep  1 08:18 around_the_world_in_80_days.txt
-rw-r--r-- 1 jovyan jovyan 462K Aug 18 20:23 le_tour_du_monde_en_quatre_vingts_jours.txt


# 1. Word Count

Instructions:
For each cell marked "double-click and add explanation here" please answer the question in your own words.
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [2]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
# Defind the rdd
# NOTE: '/content/...' (Colab path) replaced by paths relative to the notebook.
rdd = sc.textFile('around_the_world_in_80_days.txt')
rdd_2 = sc.textFile('le_tour_du_monde_en_quatre_vingts_jours.txt')

print("English lines:", rdd.count())
print("French  lines:", rdd_2.count())

English lines: 8312


French  lines: 9969


In [4]:
# view the first x lines of the rdd
rdd.take(20)

['The Project Gutenberg eBook of Around the World in Eighty Days',
 '    ',
 'This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms',
 'of the Project Gutenberg License included with this eBook or online',
 'at www.gutenberg.org. If you are not located in the United States,',
 'you will have to check the laws of the country where you are located',
 'before using this eBook.',
 '',
 'Title: Around the World in Eighty Days',
 '',
 'Author: Jules Verne',
 '',
 'Translator: George M. Towle',
 '',
 '',
 '        ',
 'Release date: January 1, 1994 [eBook #103]',
 '                Most recently updated: October 29, 2024']

In [5]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [6]:
# Note and explain the output of the below command
words

PythonRDD[7] at RDD at PythonRDD.scala:59

**Explanation — why `words` shows no word at all**

The output is something like `PythonRDD[3] at RDD at PythonRDD.scala:53`: an
object description, not data.

`flatMap` is a **transformation**, and every Spark transformation is **lazy**.
Evaluating that cell did not read a single byte of the text file. All Spark did
was add a node to the *lineage* (the DAG of dependencies): "this RDD is
obtained by applying this lambda to every element of the RDD built from
`around_the_world_in_80_days.txt`".

So `words` is a *recipe* — a handle on a distributed, partitioned,
not-yet-computed collection. What gets printed is just its `repr`: the
implementation class (`PythonRDD`, because the lambda has to run in a Python
worker alongside the JVM executor), its id in the DAG, and where it was
created. Nothing actually runs until an **action** asks for a concrete
result.

In [7]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
words.collect()

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for',
 'the',
 'use',
 'of',
 'anyone',
 'anywhere',
 'in',
 'the',
 'United',
 'States',
 'and',
 'most',
 'other',
 'parts',
 'of',
 'the',
 'world',
 'at',
 'no',
 'cost',
 'and',
 'with',
 'almost',
 'no',
 'restrictions',
 'whatsoever.',
 'You',
 'may',
 'copy',
 'it,',
 'give',
 'it',
 'away',
 'or',
 're-use',
 'it',
 'under',
 'the',
 'terms',
 'of',
 'the',
 'Project',
 'Gutenberg',
 'License',
 'included',
 'with',
 'this',
 'eBook',
 'or',
 'online',
 'at',
 'www.gutenberg.org.',
 'If',
 'you',
 'are',
 'not',
 'located',
 'in',
 'the',
 'United',
 'States,',
 'you',
 'will',
 'have',
 'to',
 'check',
 'the',
 'laws',
 'of',
 'the',
 'country',
 'where',
 'you',
 'are',
 'located',
 'before',
 'using',
 'this',
 'eBook.',
 '',
 'Title:',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 'Author:',
 'Jules'

**Explanation — what `.collect()` changes**

`collect()` is an **action**, not a transformation, and it is the trigger:
Spark finally turns the lineage into a physical plan, splits the file into
partitions, ships the Python lambda to the executors, runs `split(' ')` on
every line in parallel, then **pulls every resulting element back to the
driver** and materialises them as an ordinary Python `list` of `str`.

Two differences with the previous cell:

* **Laziness vs. execution** — the previous cell *described* a computation,
  this one *performs* it. Only now is the file actually read.
* **Distributed handle vs. local data** — `words` lives across the executors;
  the result of `words.collect()` lives entirely in the driver's memory as a
  plain Python object. It is no longer an RDD, so no Spark transformation can
  be applied to it any more.

That second point is also why `collect()` is the one action to be suspicious
of: it has no upper bound. Here the book is ~400 KB so ~68k strings come back
and it is harmless, but `collect()` on a multi-GB RDD kills the driver with an
`OutOfMemoryError`. `take(n)`, `first()`, `count()`, `takeSample()` or writing
straight to storage are the bounded alternatives — `take(20)` only computes as
many partitions as it needs to gather 20 elements, rather than the whole
dataset.

In [8]:
# nicer print
# NOTE: sliced to 50. The full collect() prints ~68k lines and makes the
# notebook file unreadable; drop the [:50] to print every word.
for w in words.collect()[:50]:
    print(w)

The
Project
Gutenberg
eBook
of
Around
the
World
in
Eighty
Days





This
eBook
is
for
the
use
of
anyone
anywhere
in
the
United
States
and
most
other
parts
of
the
world
at
no
cost
and
with
almost
no
restrictions
whatsoever.
You
may
copy
it,
give


In [9]:
# Print first x words
words.take(20)

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for']

In [10]:
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.

# `?` is the IPython help magic: it shows the signature and the docstring of an
# object without leaving the notebook.
rdd.flatMap?

# printed version too, so the answer stays visible in the saved notebook
help(rdd.flatMap)

Help on method flatMap in module pyspark.core.rdd:

flatMap(f: Callable[[~T], Iterable[~U]], preservesPartitioning: bool = False) -> 'RDD[U]' method of pyspark.core.rdd.RDD instance
    Return a new RDD by first applying a function to all elements of this
    RDD, and then flattening the results.

    .. versionadded:: 0.7.0

    Parameters
    ----------
    f : function
        a function to turn a T into a sequence of U
    preservesPartitioning : bool, optional, default False
        indicates whether the input function preserves the partitioner,
        which should be False unless this is a pair RDD and the input
        function doesn't modify the keys

    Returns
    -------
    :class:`RDD`
        a new :class:`RDD` by applying a function to all elements

    See Also
    --------
    :meth:`RDD.map`
    :meth:`RDD.mapPartitions`
    :meth:`RDD.mapPartitionsWithIndex`
    :meth:`RDD.mapPartitionsWithSplit`

    Examples
    --------
    >>> rdd = sc.parallelize([2, 3, 4])
  

**Explanation — `map` vs `flatMap`**

The docstring says: *"Return a new RDD by first applying a function to all
elements of this RDD, and then flattening the results."* Two steps, and the
second one is the whole point.

The lambda `lambda lines: lines.split(' ')` receives **one line** and returns a
**Python list of words**. With `map` we would get one list per line, i.e. an
RDD of lists — a nested, 1-to-1 structure:

```
"Around the world"  ->  ["Around", "the", "world"]   # map     : 1 line -> 1 list
"Around the world"  ->  "Around", "the", "world"     # flatMap : 1 line -> 3 elements
```

`flatMap` unpacks (flattens) each returned iterable and concatenates them, so
the resulting RDD has one element **per word** instead of one per line. It is a
1-to-N transformation: an input element can produce many outputs, one, or none
at all — returning `[]` simply drops it, which makes `flatMap` a mapper and a
filter at the same time.

This is what makes the word count possible at all: `reduceByKey` needs to group
individual words, and it could never do that on an RDD whose elements are
lists.

In [11]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

# NOTE: sliced to 50 for the same readability reason as above.
for w in words.collect()[:50]:
    print(w)

('The', 1)
('Project', 1)
('Gutenberg', 1)
('eBook', 1)
('of', 1)
('Around', 1)
('the', 1)
('World', 1)
('in', 1)
('Eighty', 1)
('Days', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('This', 1)
('eBook', 1)
('is', 1)
('for', 1)
('the', 1)
('use', 1)
('of', 1)
('anyone', 1)
('anywhere', 1)
('in', 1)
('the', 1)
('United', 1)
('States', 1)
('and', 1)
('most', 1)
('other', 1)
('parts', 1)
('of', 1)
('the', 1)
('world', 1)
('at', 1)
('no', 1)
('cost', 1)
('and', 1)
('with', 1)
('almost', 1)
('no', 1)
('restrictions', 1)
('whatsoever.', 1)
('You', 1)
('may', 1)
('copy', 1)
('it,', 1)
('give', 1)


## a. Count the occurrence of each word

`map` puts every word into a `(key, value)` pair with a count of 1, then
`reduceByKey` merges all pairs sharing the same key with the associative
function `a + b`.

`reduceByKey` is preferred over `groupByKey().map(len)` here: it combines
values *locally on each partition first* (map-side combine), so it only
shuffles one partial count per distinct word per partition instead of every
single occurrence.

In [12]:
# a. count the occurence of each word
counts_raw = (rdd
              .flatMap(lambda line: line.split(' '))
              .map(lambda word: (word, 1))
              .reduceByKey(lambda a, b: a + b))

print("distinct 'words' (raw, no cleaning):", counts_raw.count())
counts_raw.take(20)

distinct 'words' (raw, no cleaning): 12610


[('Gutenberg', 60),
 ('eBook', 6),
 ('of', 1875),
 ('Around', 4),
 ('', 2193),
 ('for', 407),
 ('use', 16),
 ('anyone', 6),
 ('United', 23),
 ('States', 10),
 ('and', 1793),
 ('most', 43),
 ('other', 59),
 ('world', 30),
 ('at', 576),
 ('no', 124),
 ('cost', 9),
 ('with', 550),
 ('almost', 19),
 ('restrictions', 2)]

## b. Change all capital letters to lower case

Without this, `The`, `the` and `THE` are three different keys. Lower-casing
happens **before** the pair is built, so the counts of the variants collapse
into a single key.

In [13]:
# b. a common first step in text analysis, change all capital letters to lower case
counts_lower = (rdd
                .flatMap(lambda line: line.lower().split(' '))
                .map(lambda word: (word, 1))
                .reduceByKey(lambda a, b: a + b))

print("distinct before lowercasing:", counts_raw.count())
print("distinct after  lowercasing:", counts_lower.count())
counts_lower.take(20)

distinct before lowercasing: 12610


distinct after  lowercasing: 11884


[('of', 1926),
 ('around', 32),
 ('world', 35),
 ('eighty', 27),
 ('days', 46),
 ('', 2193),
 ('this', 341),
 ('for', 414),
 ('use', 19),
 ('anyone', 6),
 ('united', 27),
 ('states', 14),
 ('and', 1835),
 ('most', 45),
 ('other', 63),
 ('at', 645),
 ('no', 137),
 ('cost', 12),
 ('with', 562),
 ('almost', 19)]

## c. Eliminate the stop words

Stop words are the grammatical glue of a language (`the`, `of`, `and`, ...).
They are by far the most frequent tokens but say almost nothing about what a
text is *about*, so they drown out the interesting vocabulary.

The list below is a standard English stop-word list, written as a `frozenset`
so the `in` test is O(1) rather than O(n) — that test runs once per word, about
68 000 times per pass.

In [14]:
# c. eliminate the stop words.
STOP_WORDS = frozenset({
    "a", "about", "above", "after", "again", "against", "all", "am", "an",
    "and", "any", "are", "aren't", "as", "at", "be", "because", "been",
    "before", "being", "below", "between", "both", "but", "by", "can",
    "can't", "cannot", "could", "couldn't", "did", "didn't", "do", "does",
    "doesn't", "doing", "don't", "down", "during", "each", "few", "for",
    "from", "further", "had", "hadn't", "has", "hasn't", "have", "haven't",
    "having", "he", "he'd", "he'll", "he's", "her", "here", "here's", "hers",
    "herself", "him", "himself", "his", "how", "how's", "i", "i'd", "i'll",
    "i'm", "i've", "if", "in", "into", "is", "isn't", "it", "it's", "its",
    "itself", "let's", "me", "more", "most", "mustn't", "my", "myself", "no",
    "nor", "not", "of", "off", "on", "once", "only", "or", "other", "ought",
    "our", "ours", "ourselves", "out", "over", "own", "same", "shan't", "she",
    "she'd", "she'll", "she's", "should", "shouldn't", "so", "some", "such",
    "than", "that", "that's", "the", "their", "theirs", "them", "themselves",
    "then", "there", "there's", "these", "they", "they'd", "they'll",
    "they're", "they've", "this", "those", "through", "to", "too", "under",
    "until", "up", "very", "was", "wasn't", "we", "we'd", "we'll", "we're",
    "we've", "were", "weren't", "what", "what's", "when", "when's", "where",
    "where's", "which", "while", "who", "who's", "whom", "why", "why's",
    "with", "won't", "would", "wouldn't", "you", "you'd", "you'll", "you're",
    "you've", "your", "yours", "yourself", "yourselves",
    # very frequent in 19th-century prose, and just as uninformative
    "s", "t", "said", "upon", "shall", "must", "may", "one", "two", "will",
})

counts_nostop = (rdd
                 .flatMap(lambda line: line.lower().split(' '))
                 .filter(lambda word: word not in STOP_WORDS)
                 .map(lambda word: (word, 1))
                 .reduceByKey(lambda a, b: a + b))

print("distinct before removing stop words:", counts_lower.count())
print("distinct after  removing stop words:", counts_nostop.count())
counts_nostop.take(20)

distinct before removing stop words: 11884


distinct after  removing stop words: 11755


[('around', 32),
 ('world', 35),
 ('eighty', 27),
 ('days', 46),
 ('', 2193),
 ('use', 19),
 ('anyone', 6),
 ('united', 27),
 ('states', 14),
 ('cost', 12),
 ('almost', 19),
 ('restrictions', 2),
 ('give', 18),
 ('re-use', 2),
 ('license', 12),
 ('online', 4),
 ('www.gutenberg.org.', 4),
 ('states,', 7),
 ('country', 18),
 ('using', 6)]

## d. Sort in alphabetical order

The RDD is already a pair RDD keyed by the word, so `sortByKey()` sorts it
alphabetically. `sortByKey` is a *transformation* (it returns an RDD) but,
unlike `map`, it is not fully lazy: it has to sample the keys to build the
range partitioner, so it launches a job straight away.

`take(20)` after a sort is cheap — the data is range-partitioned, so the first
20 elements all come from the first partition.

In [15]:
# d. sort in alphabetical order
alphabetical_nostop = counts_nostop.sortByKey()
alphabetical_nostop.take(20)

[('', 2193),
 ('#103]', 1),
 ('#516,', 1),
 ('$5,000)', 1),
 ('&c.,', 1),
 ('($1', 1),
 ('(862)', 1),
 ('(a)', 1),
 ('(and', 1),
 ('(any', 1),
 ('(b)', 1),
 ('(c)', 1),
 ('(does', 1),
 ('(if', 1),
 ('(japan),', 1),
 ('(or', 3),
 ('(saturday,', 1),
 ('(sort', 1),
 ('(sunday)', 1),
 ('(trademark/copyright)', 1)]

## e. Sort descending by word frequency

`sortBy` takes the value out of each pair as the sort key. Two ways:

* `sortBy(lambda kv: kv[1], ascending=False)` — a full distributed sort; the
  whole RDD stays available in sorted order.
* `takeOrdered(n, key=lambda kv: -kv[1])` — an action, no full shuffle: each
  partition keeps only its local top *n* and those are merged in the driver.
  Much cheaper when only the head of the ranking is needed.

In [16]:
# e. sort descending by word frequency
by_frequency = counts_nostop.sortBy(lambda kv: kv[1], ascending=False)
by_frequency.take(20)

[('', 2193),
 ('mr.', 373),
 ('fogg', 365),
 ('phileas', 250),
 ('passepartout', 239),
 ('fogg,', 132),
 ('fix', 129),
 ('passepartout,', 121),
 ('“i', 115),
 ('hundred', 92),
 ('now', 92),
 ('replied', 89),
 ('project', 87),
 ('without', 86),
 ('thousand', 83),
 ('and,', 83),
 ('time', 82),
 ('made', 81),
 ('like', 80),
 ('chapter', 74)]

In [17]:
# the same top 20 without a full sort: cheaper, and what should be used when
# only the head of the ranking matters
counts_nostop.takeOrdered(20, key=lambda kv: -kv[1])

[('', 2193),
 ('mr.', 373),
 ('fogg', 365),
 ('phileas', 250),
 ('passepartout', 239),
 ('fogg,', 132),
 ('fix', 129),
 ('passepartout,', 121),
 ('“i', 115),
 ('hundred', 92),
 ('now', 92),
 ('replied', 89),
 ('project', 87),
 ('without', 86),
 ('thousand', 83),
 ('and,', 83),
 ('time', 82),
 ('made', 81),
 ('like', 80),
 ('chapter', 74)]

## f. Remove punctuations and blank spaces

Two distinct problems needing two different fixes:

* **Blank tokens.** `split(' ')` splits on *exactly one* space character, so a
  double space, an indent or a tab yields empty strings `''`. Calling `split()`
  with no argument splits on any run of whitespace and discards the empties —
  one call fixes it.
* **Punctuation glued to words.** `world.`, `"Passepartout` and `Fogg,` are
  three keys distinct from `world`, `passepartout` and `fogg`. `str.strip()`
  removes leading and trailing characters only, which is deliberately better
  than stripping punctuation everywhere: it keeps `don't` and `well-known`
  intact instead of turning them into `dont` and `wellknown`.

Two details specific to this text: Gutenberg uses typographic quotes
(`“ ” ‘ ’`) and dashes that are **not** in `string.punctuation`, and the em
dash is often glued between two words (`Fogg—who`), so it must become a
separator *before* splitting rather than merely be stripped.

In [18]:
# f. remove punctuations and blank spaces
import string

# string.punctuation is ASCII only; Gutenberg texts use typographic variants
PUNCTUATION = string.punctuation + '“”‘’«»—–…‹›„'
# characters that glue two words together, so they must become separators
SEPARATORS = str.maketrans({'—': ' ', '–': ' ', '…': ' ', '\u00a0': ' '})

cleaned = (rdd
           .flatMap(lambda line: line.lower().translate(SEPARATORS).split())
           .map(lambda word: word.strip(PUNCTUATION))
           .filter(lambda word: word and word not in STOP_WORDS)
           .map(lambda word: (word, 1))
           .reduceByKey(lambda a, b: a + b))

print("distinct words, fully cleaned:", cleaned.count())
cleaned.takeOrdered(20, key=lambda kv: -kv[1])

distinct words, fully cleaned: 7268


[('fogg', 602),
 ('passepartout', 404),
 ('mr', 391),
 ('phileas', 256),
 ('fix', 240),
 ('time', 126),
 ('aouda', 125),
 ('train', 119),
 ('now', 110),
 ('master', 104),
 ('sir', 103),
 ('project', 99),
 ('hundred', 98),
 ('replied', 93),
 ('hours', 90),
 ('without', 88),
 ('thousand', 88),
 ('steamer', 88),
 ('made', 87),
 ('day', 85)]

## Everything chained into a single function

The order of the steps is not arbitrary — it is chosen so Spark does the least
possible work:

1. `flatMap` — lower-case, un-glue and tokenise in **one** pass per line.
   Doing `.map(str.lower)` on the words afterwards would be a second pass over
   68 000 elements instead of one pass over 12 000 lines.
2. `map(strip)` — normalise *before* filtering, so that `"the` is recognised as
   the stop word `the` and dropped rather than counted.
3. `filter` — drop blanks and stop words **before** the shuffle. Filtering
   early is the single most valuable optimisation here: roughly 45 % of the
   tokens never reach `reduceByKey`, so they are never serialised nor sent over
   the network.
4. `map` to pairs, then `reduceByKey` — the only wide (shuffling) step.
5. Sorting is deliberately left **outside** the counting function. It is a
   second shuffle, only worth paying for when the caller actually wants ordered
   output — which is why `top_n` uses `takeOrdered` instead.

In [19]:
# Everything above, chained into one reusable function.
import string

PUNCTUATION = string.punctuation + '“”‘’«»—–…‹›„'
SEPARATORS = str.maketrans({'—': ' ', '–': ' ', '…': ' ', '\u00a0': ' '})


def word_count(lines, stop_words=STOP_WORDS,
               punctuation=PUNCTUATION, separators=SEPARATORS):
    """Count words in an RDD of text lines.

    Lower-cases, splits on whitespace, strips punctuation from both ends of
    every token, drops blanks and stop words, then counts.

    Returns a pair RDD of (word, count) - unsorted, because sorting is a
    second shuffle the caller may not want.
    """
    return (lines
            .flatMap(lambda line: line.lower()
                                      .translate(separators)
                                      .split())
            .map(lambda word: word.strip(punctuation))
            .filter(lambda word: word and word not in stop_words)
            .map(lambda word: (word, 1))
            .reduceByKey(lambda a, b: a + b))


def top_n(counts, n=20):
    """Most frequent words first, ties broken alphabetically."""
    return counts.takeOrdered(n, key=lambda kv: (-kv[1], kv[0]))


def alphabetical(counts):
    """The same counts, sorted A->Z."""
    return counts.sortByKey()


counts = word_count(rdd).cache()   # cached: reused by the cells below
print("distinct words:", counts.count())
print("total words   :", counts.map(lambda kv: kv[1]).sum())

distinct words: 7268
total words   : 33241


In [20]:
# steps d. and e. applied to the final, fully cleaned result
print("--- alphabetical (first 20) ---")
for word, n in alphabetical(counts).take(20):
    print(f"{n:5d}  {word}")

print("\n--- most frequent (top 20) ---")
for word, n in top_n(counts, 20):
    print(f"{n:5d}  {word}")

--- alphabetical (first 20) ---


    1  07042
    5  1
    1  1,000
    1  1,170
    1  1.a
    1  1.b
    2  1.c
    1  1.d
    2  1.e
    5  1.e.1
    1  1.e.2
    1  1.e.3
    1  1.e.4
    1  1.e.5
    1  1.e.6
    3  1.e.7
    4  1.e.8
    3  1.e.9
    1  1.f
    1  1.f.1

--- most frequent (top 20) ---
  602  fogg
  404  passepartout
  391  mr
  256  phileas
  240  fix
  126  time
  125  aouda
  119  train
  110  now
  104  master
  103  sir
   99  project
   98  hundred
   93  replied
   90  hours
   88  steamer
   88  thousand
   88  without
   87  made
   85  day


# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [21]:
# Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
  # Re-shape (name, age) into (name, (age, 1)): the name becomes the key
  # and the value becomes a 2-slot accumulator holding a partial sum of
  # ages plus the number of rows that fed that sum. The count must be
  # carried explicitly, because the next step only ever sees values.
  .map(lambda x: (x[0], (x[1], 1)))
  # Merge, per key, the accumulators pairwise: sums with sums, counts
  # with counts. The lambda is associative and commutative, so Spark can
  # combine locally on each partition before the shuffle and send only
  # one partial (sum, count) per name per partition over the network.
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # Collapse each (name, (total_age, n_rows)) into (name, mean_age) by
  # dividing the total by the count. This projection turns the
  # accumulator back into the value we actually wanted.
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

agesRDD.collect()

[('Jules', 30.0), ('TD', 35.0), ('Brooke', 22.5), ('Denny', 31.0)]

**In one sentence:** it computes the **average age per name** — `sum /
count` per key, done the map-reduce way.

`Brooke` appears twice (20 and 25) so she averages `45 / 2 = 22.5`; every other
name appears once, so its average is simply its own age.

Why not the obvious `reduceByKey(lambda a, b: (a + b) / 2)`? Because averaging
is **not associative**. With three values it would give `((a + b) / 2 + c) / 2`,
weighting `c` as half the result instead of a third — and since Spark is free
to choose the order and the grouping of the merges, the answer would depend on
the partitioning. The `(sum, count)` accumulator works precisely because
addition *is* associative; the division happens exactly once, at the very end,
in the final `map`.

This is the standard pattern for any non-associative aggregate in a distributed
engine, and it is what `AVG()` compiles down to in Spark SQL.

# Where to go from here

Further exploration for students who complete the lab before the end of the session or want to go further.

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two
- recomplete the exercises using a the docker install
- install java and spark directly onto host machine and either rexplore this notebook or perform eda on other data sets
- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible

## 3.1 Stripping the Project Gutenberg boilerplate

Every Gutenberg file is wrapped in a licence header and footer of a few hundred
lines. Those words (`gutenberg`, `ebook`, `license`, `donations`, ...) are not
part of the novel and pollute the counts.

`textFile` gives no line numbers, so `zipWithIndex` attaches them, the marker
positions are located, and the range in between is kept. Note that `first()` is
an action: both bounds are ordinary Python integers by the time they are used
in the closure, and Spark ships them to the executors inside the serialised
lambda.

In [22]:
def strip_gutenberg(lines):
    """Keep only the lines between the START and END Gutenberg markers."""
    indexed = lines.zipWithIndex().cache()

    def marker(tag):
        hits = indexed.filter(lambda li: li[0].startswith(f"*** {tag} OF"))
        return hits.map(lambda li: li[1]).first()

    start, end = marker("START"), marker("END")
    body = (indexed
            .filter(lambda li: start < li[1] < end)
            .map(lambda li: li[0]))
    indexed.unpersist()
    return body


book = strip_gutenberg(rdd).cache()
print(f"raw file : {rdd.count():>6} lines")
print(f"novel    : {book.count():>6} lines")

counts_book = word_count(book).cache()
print("\n--- top 20, boilerplate removed ---")
for word, n in top_n(counts_book, 20):
    print(f"{n:5d}  {word}")

raw file :   8312 lines
novel    :   7934 lines



--- top 20, boilerplate removed ---


  602  fogg
  404  passepartout
  391  mr
  256  phileas
  239  fix
  126  time
  125  aouda
  119  train
  110  now
  104  master
  103  sir
   98  hundred
   93  replied
   90  hours
   88  steamer
   88  thousand
   87  made
   85  day
   84  well
   81  like


## 3.2 Comparing the English and the French edition

The French text needs its own stop-word list and its own separator handling:
elisions (`l'homme`, `qu'il`, `d'un`) are glued by an apostrophe, so for French
the apostrophe has to become a separator — while for English it must *not*,
otherwise `don't` breaks into `don` and `t`.

This is exactly why `word_count` takes its stop words, punctuation and
separators as **parameters** instead of hard-coding them: one function, both
languages.

In [23]:
STOP_WORDS_FR = frozenset({
    "a", "à", "ai", "aux", "au", "avec", "avait", "avaient", "aussi", "bien",
    "car", "ce", "cela", "ces", "cet", "cette", "ceux", "chez", "comme",
    "dans", "de", "des", "du", "dont", "deux", "elle", "elles", "en",
    "encore", "est", "et", "étaient", "était", "être", "eu", "fait", "faire",
    "furent", "fut", "il", "ils", "je", "la", "le", "les", "leur", "leurs",
    "lui", "là", "ma", "mais", "me", "même", "mes", "moi", "mon", "n", "ne",
    "ni", "nous", "on", "ont", "ou", "où", "par", "pas", "peu", "plus",
    "pour", "pu", "qu", "quand", "que", "quel", "quelle", "qui", "quoi",
    "sa", "sans", "se", "ses", "si", "sont", "son", "sous", "sur", "ta",
    "te", "tes", "toi", "ton", "tous", "tout", "toute", "toutes", "très",
    "tu", "un", "une", "vous", "y", "d", "l", "s", "c", "j", "m", "t",
    "dit", "avoir", "cependant", "alors", "donc", "ainsi", "vers", "point",
})

# in French the apostrophe glues words together, so it becomes a separator
SEPARATORS_FR = str.maketrans(
    {'—': ' ', '–': ' ', '…': ' ', '\u00a0': ' ', "'": ' ', '’': ' '})

book_fr = strip_gutenberg(rdd_2).cache()
counts_fr = word_count(book_fr,
                       stop_words=STOP_WORDS_FR,
                       separators=SEPARATORS_FR).cache()

en_total = counts_book.map(lambda kv: kv[1]).sum()
fr_total = counts_fr.map(lambda kv: kv[1]).sum()

print(f"{'':<12} {'EN':>10} {'FR':>10}")
print(f"{'lines':<12} {book.count():>10} {book_fr.count():>10}")
print(f"{'words kept':<12} {en_total:>10} {fr_total:>10}")
print(f"{'vocabulary':<12} {counts_book.count():>10} {counts_fr.count():>10}")

print(f"\n{'EN':<20}{'FR':<20}")
for (we, ne), (wf, nf) in zip(top_n(counts_book, 15), top_n(counts_fr, 15)):
    print(f"{we:<14}{ne:>5}   {wf:<14}{nf:>5}")

                     EN         FR


lines              7934       9585
words kept        31556      36711


vocabulary         6935       9131

EN                  FR                  


fogg            602   fogg            689
passepartout    404   passepartout    460
mr              391   phileas         332
phileas         256   mr              287
fix             239   fix             286
time            126   heures          243
aouda           125   répondit        215
train           119   monsieur        145
now             110   aouda           136
master          104   après           132
sir             103   mrs             131
hundred          98   quelques        129
replied          93   maître          123
hours            90   été             120
steamer          88   train           115


**Reading the comparison**

The proper nouns line up almost perfectly across the two editions — `fogg`,
`passepartout`, `fix`, `aouda`. That is the sanity check one wants: it is the
same novel, and names survive translation. Their *counts* differ mostly because
of how each language handles elision and pronouns, not because of the story.

The French vocabulary is larger for a comparable number of tokens. That is not
a richer text, it is morphology: French inflects verbs and agrees adjectives in
gender and number, so `parti / partie / partis / parties` are four keys where
English has one. A serious comparison would need lemmatisation (`nltk`,
`spaCy`), which is out of scope for a pure-RDD lab — worth flagging as a limit
of the numbers above rather than pretending they settle anything.

## 3.3 Timing: does the order of the steps matter?

A small timer, then the same word count written three ways. Each variant is
forced with an action (`count()`), and `book` is cached first so the disk read
is not part of the measurement.

The interesting comparison is **filter before vs. after the shuffle**: same
result, very different amount of work.

In [24]:
import time


def timeit(label, fn, repeat=3):
    """Run fn `repeat` times, print and return the best wall-clock time."""
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        result = fn()
        best = min(best, time.perf_counter() - t0)
    print(f"{label:<30} {best*1000:8.0f} ms   -> {result}")
    return best


book.count()  # warm the cache, so the file read is not measured


def v1_filter_early():
    """Drop blanks and stop words BEFORE the shuffle (the good one)."""
    return (book
            .flatMap(lambda l: l.lower().translate(SEPARATORS).split())
            .map(lambda w: w.strip(PUNCTUATION))
            .filter(lambda w: w and w not in STOP_WORDS)
            .map(lambda w: (w, 1))
            .reduceByKey(lambda a, b: a + b)
            .count())


def v2_filter_late():
    """Same result, but stop words get shuffled and are dropped after."""
    return (book
            .flatMap(lambda l: l.lower().translate(SEPARATORS).split())
            .map(lambda w: w.strip(PUNCTUATION))
            .map(lambda w: (w, 1))
            .reduceByKey(lambda a, b: a + b)
            .filter(lambda kv: kv[0] and kv[0] not in STOP_WORDS)
            .count())


def v3_groupbykey():
    """No map-side combine: every occurrence crosses the network."""
    return (book
            .flatMap(lambda l: l.lower().translate(SEPARATORS).split())
            .map(lambda w: w.strip(PUNCTUATION))
            .filter(lambda w: w and w not in STOP_WORDS)
            .map(lambda w: (w, 1))
            .groupByKey()
            .map(lambda kv: (kv[0], sum(kv[1])))
            .count())


t1 = timeit("filter early + reduceByKey", v1_filter_early)
t2 = timeit("filter late  + reduceByKey", v2_filter_late)
t3 = timeit("filter early + groupByKey", v3_groupbykey)

print(f"\nfilter late is {t2/t1:.2f}x the best time")
print(f"groupByKey  is {t3/t1:.2f}x the best time")

filter early + reduceByKey          265 ms   -> 6935


filter late  + reduceByKey          288 ms   -> 6935


filter early + groupByKey           264 ms   -> 6935

filter late is 1.09x the best time
groupByKey  is 1.00x the best time


**What the timings do and do not show**

On a 400 KB book running on a single local machine, the three variants land
within noise of each other, and `groupByKey` may even look perfectly fine. That
is the honest reading of the numbers printed above: this dataset is far too
small for the shuffle to dominate, and JVM warm-up plus Python worker start-up
cost more than the computation itself.

The reason to still prefer variant 1 is a property of the *plan*, not of this
measurement:

* **filter early** removes roughly 45 % of the tokens *before* they are
  serialised, hashed and sent across the network. The saving is proportional to
  the shuffle volume — invisible here, decisive at terabyte scale.
* **`groupByKey`** has no map-side combine: it materialises the full list of
  every occurrence of a key on a single executor. For `the` in a large corpus
  that is millions of elements in one partition, and it is the classic cause of
  an executor OOM. `reduceByKey` never holds more than one accumulator per key.

That is the lesson worth keeping from this lab: on RDDs the shape of the DAG is
chosen by the programmer, and a bad shape does not fail loudly on a small
dataset — it fails in production. DataFrames largely remove the trap by letting
Catalyst push filters down on its own, which is exactly why they are the
default API today.

In [25]:
# stop the session and release the executors
spark.stop()